In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# read enrichment data
rich = pd.read_csv('', dtype={'id_number' : str,
                                             'cellphone_number' : str})

In [ ]:
# first sheet of general support service
df1 = (
    pd.read_excel(
        io='',
        sheet_name=0,
        usecols=[1, 2, 4, 5, 6],
        skiprows=2,
        names=['umuzi_email', 'cohort_name', 'service_used', 'service_name', 'date_service_accessed']
    )
    .assign(
        umuzi_email = lambda df: df['umuzi_email'].str.split(",")
    )
    .explode("umuzi_email", ignore_index=True)
    .assign(
        umuzi_email = lambda df: df['umuzi_email'].str.strip(),
        cohort_name = lambda df: df['cohort_name'].str.strip(),
        date_service_accessed = lambda df: pd.to_datetime(df['date_service_accessed']).dt.date
    )
)

df1.head(10)

In [ ]:
df2 = (
    pd.read_excel(
        io="",
        sheet_name = 1,
        usecols=[2, 3, 5, 6, 7],
        names=['umuzi_email', 'cohort_name', 'service_used', 'service_name', 'date_service_accessed'],
        skiprows=2
    )
    .assign(
        date_service_accessed = lambda df: pd.to_datetime(df['date_service_accessed']).dt.date,
        umuzi_email = lambda df: df['umuzi_email'].str.split(",")
    )
    .explode("umuzi_email", ignore_index=True)
    .assign(
        umuzi_email = lambda df: df['umuzi_email'].str.strip()
    )
)

df2.sample(10)

In [ ]:
data = pd.concat([df1, df2], ignore_index=True)

In [ ]:
data.drop(
    labels=212,
    axis=0, inplace=True
)

In [ ]:
data.columns

In [ ]:
# Find emails in the data that can be matched on the 'email' column but not 'umuzi_email' column of the rich dataset
# This identifies records that need special handling during the merge (using email instead of umuzi_email)
a = set(set(data['umuzi_email'])).difference(set(rich['email']))
b = set(set(data['umuzi_email'])).difference(set(rich['umuzi_email']))

b - a

In [ ]:
len(b)

> Confirmation proves that the 22 are not of South African origin. They are therefore not viable to be reported on.

In [ ]:
data.drop(
    labels=data.loc[data['umuzi_email'].isin(b)].index,
    axis=0, inplace=True
)

In [ ]:
premerge_rows = data.shape[0]

In [ ]:
data = data.merge(
    rich.drop(columns=['email']), 
    on='umuzi_email', 
    how='left')

In [ ]:
assert premerge_rows == data.shape[0], "Duplicates introduced after merge due to non-unique merge keys."

In [ ]:
data['learner_id'] = data['learner_id'].astype(int)

In [ ]:
data['date_service_accessed'] = pd.to_datetime(data['date_service_accessed']).dt.date

In [ ]:
data['age_service_accessed'] = (
    pd.to_datetime(data['date_service_accessed'])- pd.to_datetime(data['date_of_birth'])
).dt.days // 365

# add age bins
data['age_range'] = pd.cut(
    x=data['age_service_accessed'],
    bins=[-np.inf, 17, 25, 35, np.inf],
    labels=['17 and below', '18-25', '26-35', '36+']
)

In [ ]:
data['month_of_service_accessed'] = pd.to_datetime(data['date_service_accessed']).dt.month_name()

data['month_of_service_accessed'] = data['month_of_service_accessed'] + ' 2025'

In [ ]:
data = data[[
    'learner_id', 'application_id', 'umuzi_email',
    'first_name', 'last_name', 'cohort_name', 'cellphone_number',
    'id_number', 'gender', 'date_of_birth', 'age_service_accessed', 'age_range', 'race',
    'residential_area_type', 'has_disability_or_differently_abled',
    'application_date', 'nearest_metro', 'province', 'date_service_accessed',
    'month_of_service_accessed', 'service_used', 'service_name'
]]

In [ ]:
# export to ind7 data

# data.to_csv('../Sink Datasets/indicator_7_placements.csv', index=False)

In [ ]:
# append to existing ind7 data
# data.to_csv('../Sink Datasets/indicator_7_data.csv', index=False, mode='a', header=False)